# Raw-direct comparison arm: 830,000 PPO timesteps
Select **Runtime > Change runtime type > GPU > GPU type: L4**, then Run all. This arm feeds observations straight to the policy with no learned extractor, and stops on the timestep ceiling rather than a wall clock. Drive artifacts are authoritative and rerunning all resumes verified state.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
EXPERIMENT_ROOT = Path("/content/drive/MyDrive/CNN-RL-improved/raw-direct-830k-seed0")
PPO_ROOT = EXPERIMENT_ROOT / "ppo"
TARGET_TIMESTEPS = 830_000
for path in (EXPERIMENT_ROOT, PPO_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print("target timesteps:", TARGET_TIMESTEPS, "->", PPO_ROOT)


In [ ]:
import os, psutil, shutil, subprocess, torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required; select an L4 GPU runtime")
GPU_NAME = torch.cuda.get_device_name(0)
ACCEPTED_GPU_NAMES = {"NVIDIA L4"}
if GPU_NAME not in ACCEPTED_GPU_NAMES:
    raise RuntimeError(f"Expected NVIDIA L4, got {GPU_NAME!r}")
if os.cpu_count() is None or not os.cpu_count() >= 2:
    raise RuntimeError("At least two visible CPU cores are required")
ram = psutil.virtual_memory()
drive_space = shutil.disk_usage("/content/drive/MyDrive")
if ram.total < 10 * 1024**3:
    raise RuntimeError("At least 10 GiB system RAM is required")
if drive_space.free < 10 * 1024**3:
    raise RuntimeError("At least 10 GiB free Drive space is required")
subprocess.run(["nvidia-smi"], check=True)
print({"gpu": GPU_NAME, "torch": torch.__version__, "cuda": torch.version.cuda, "cpu_cores": os.cpu_count(), "ram_gib": ram.total / 1024**3, "drive_free_gib": drive_space.free / 1024**3})


In [ ]:
import subprocess
REPOSITORY = "https://github.com/LMS4681/CNN-RL-Raw-Comparison.git"
RELEASE_TAG = "raw-direct-830k-v1"
target = Path("/content/CNN-RL-Raw-Comparison").resolve()
if target.exists():
    if target.parent != Path("/content") or not (target / ".git").is_dir():
        raise RuntimeError(f"Unsafe or non-git checkout path: {target}")
else:
    subprocess.run(["git", "clone", "--branch", RELEASE_TAG, "--depth", "1", REPOSITORY, str(target)], check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=target, text=True).strip()
tag_head = subprocess.check_output(["git", "rev-list", "-n", "1", RELEASE_TAG], cwd=target, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=target, text=True)
if head != tag_head or dirty:
    raise RuntimeError(f"Checkout is not the clean pinned tag: HEAD={head}, tag={tag_head}, dirty={dirty!r}")
print("Pinned checkout:", head)


In [ ]:
import importlib.metadata, json, subprocess, sys
def child_torch_snapshot():
    source = 'import json, torch; print(json.dumps({"version": torch.__version__, "file": torch.__file__, "cuda": torch.version.cuda, "available": torch.cuda.is_available()}))'
    return json.loads(subprocess.check_output([sys.executable, "-c", source], text=True))
def pip_check_conflicts():
    result = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, capture_output=True)
    return {line.strip() for line in (result.stdout + result.stderr).splitlines() if line.strip()}
before_torch = child_torch_snapshot()
before_pip_conflicts = pip_check_conflicts()
lock_path = target / "AllocRL" / "requirements-comparison.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--require-hashes", "-r", str(lock_path)], check=True)
after_torch = child_torch_snapshot()
after_pip_conflicts = pip_check_conflicts()
new_pip_conflicts = after_pip_conflicts - before_pip_conflicts
if after_torch != before_torch:
    raise RuntimeError(f"Colab Torch stack changed: {before_torch} -> {after_torch}")
if new_pip_conflicts:
    raise RuntimeError("Dependency install introduced conflicts: " + "; ".join(sorted(new_pip_conflicts)))


In [ ]:
import hashlib, json, os
ALLOC_RL = target / "AllocRL"
os.chdir(ALLOC_RL)
expected_hashes = {
    "data/fixed_eval_scenarios.json": "913cac9046dec8164ef65da60275522f7127de5ea775b1c5a6b6aac255716271",
    "data/data_split_manifest.json": "601bd6143ed8890577e5ff34921241d36fd6a0e99c4bdab4e26152ab168178f8",
    "requirements-comparison.txt": "37634576e34043d169cf24bfc0cc2261818dc65b9358d4b9b2e46ab614d0bdda",
}
for relative, expected in expected_hashes.items():
    actual = hashlib.sha256((ALLOC_RL / relative).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"Immutable input hash mismatch for {relative}: {actual} != {expected}")
print("Immutable hashes verified")


In [ ]:
# Rows a resume rolled back make the append-only curve logs regress, which would
# reject the logs on the next append. Discard exactly those rows; retained records
# keep their bytes.
import shutil, sys
if str(ALLOC_RL) not in sys.path:
    sys.path.insert(0, str(ALLOC_RL))
from comparison.training_log_validation import prune_rolled_back_rows
for curve_name, curve_kind in (("training_log.csv", "training_log"), ("loss_log.csv", "loss_log")):
    curve_path = PPO_ROOT / curve_name
    if not curve_path.is_file():
        print(f"{curve_name}: absent")
        continue
    backup_path = curve_path.with_name(curve_path.name + ".pre_prune_backup")
    if not backup_path.exists():
        shutil.copy2(curve_path, backup_path)
    discarded = prune_rolled_back_rows(curve_path, curve_kind)
    print(f"{curve_name}: {discarded} rolled-back rows discarded" if discarded else f"{curve_name}: exact")


In [ ]:
import collections, json, os, re, signal, subprocess, sys, threading, time
from datetime import datetime, timezone
from train import find_resumable_model, model_num_timesteps

# --timesteps is additive on resume, so each launch trains only the remaining budget.
resumable = find_resumable_model(PPO_ROOT)
start_timestep = int(model_num_timesteps(resumable) or 0) if resumable is not None else 0
remaining = max(TARGET_TIMESTEPS - start_timestep, 0)
print(f"durable timestep={start_timestep} target={TARGET_TIMESTEPS} remaining={remaining}")

ppo_command = [sys.executable, "-u", "train.py", "--data-dir", "./data", "--output-dir", str(PPO_ROOT), "--extractor", "raw-direct", "--state-context", "full", "--seed", "0", "--timesteps", str(remaining), "--lr", "0.0001", "--lr-schedule", "linear", "--lr-final", "0.00001", "--lr-decay-steps", "1000000", "--n-envs", "8", "--vec-env", "subproc", "--n-steps", "120", "--batch-size", "64", "--n-epochs", "5", "--gamma", "1.0", "--gae-lambda", "0.98", "--checkpoint-freq", "10000", "--holdout-eval-freq", "50000", "--holdout-selection-count", "5", "--monthly-jitter", "20", "--empirical-profile-probability", "0.2", "--device", "cuda", "--eval-scenarios", "./data/fixed_eval_scenarios.json", "--final-holdout-report", "--auto-resume", "--no-export-onnx"]
PPO_LOG_INTERVAL_SECONDS = 30
CHILD_OUTPUT_TAIL_LINES = 100
CHECKPOINT_PATTERN = re.compile(r"_(\d+)_steps\.sb3$")

def _durable_monitor_fields(ppo_root):
    checkpoints = list((Path(ppo_root) / "checkpoints").glob("*_steps.sb3"))
    steps = [int(m.group(1)) for m in (CHECKPOINT_PATTERN.search(p.name) for p in checkpoints) if m]
    if not steps:
        return [f"durable_timestep={start_timestep}", "checkpoints=0"]
    return [f"durable_timestep={max(steps)}", f"checkpoints={len(steps)}", f"target={TARGET_TIMESTEPS}"]

def _stream_child_output(stream, output_tail):
    try:
        for line in iter(stream.readline, ""):
            output_tail.append(line)
            print(line, end="", flush=True)
    finally:
        stream.close()

def _stop_training_process(training_process):
    if training_process.poll() is not None:
        return
    training_process.send_signal(signal.SIGINT)
    try:
        training_process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        training_process.terminate()
        try:
            training_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            training_process.kill()
            training_process.wait()

def _run_training_with_live_logs(command, *, cwd, env, ppo_root):
    training_process = None
    output_reader = None
    output_tail = collections.deque(maxlen=CHILD_OUTPUT_TAIL_LINES)
    try:
        print(f"[colab-monitor] starting PPO process; status interval={PPO_LOG_INTERVAL_SECONDS}s", flush=True)
        monitor_started = time.monotonic()
        training_process = subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if training_process.stdout is None:
            raise RuntimeError("PPO child output pipe was not created")
        output_reader = threading.Thread(
            target=_stream_child_output,
            args=(training_process.stdout, output_tail),
            daemon=True,
        )
        output_reader.start()
        while True:
            try:
                return_code = training_process.wait(timeout=PPO_LOG_INTERVAL_SECONDS)
                break
            except subprocess.TimeoutExpired:
                polled_return_code = training_process.poll()
                if polled_return_code is not None:
                    return_code = polled_return_code
                    break
                elapsed_seconds = int(time.monotonic() - monitor_started)
                monitor_fields = [f"elapsed_seconds={elapsed_seconds}", "process=running"]
                monitor_fields.extend(_durable_monitor_fields(ppo_root))
                print("[colab-monitor] " + " ".join(monitor_fields), flush=True)
    except KeyboardInterrupt:
        print("[colab-monitor] interrupt received; stopping PPO child", flush=True)
        if training_process is not None:
            _stop_training_process(training_process)
        raise
    finally:
        if output_reader is not None:
            output_reader.join(timeout=10)
    print(f"[colab-monitor] PPO process exited return_code={return_code}", flush=True)
    if return_code != 0:
        output_text = "".join(output_tail)
        if output_text:
            print("[colab-monitor] child output tail follows:", flush=True)
            print(output_text, end="" if output_text.endswith("\n") else "\n", flush=True)
        raise subprocess.CalledProcessError(return_code, command, output=output_text)

if remaining == 0:
    print("target already reached; skipping training")
else:
    session_started_at = datetime.now(timezone.utc)
    session_monotonic = time.monotonic()
    _run_training_with_live_logs(
        ppo_command,
        cwd=ALLOC_RL,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
        ppo_root=PPO_ROOT,
    )
    elapsed = time.monotonic() - session_monotonic
    finished = find_resumable_model(PPO_ROOT)
    end_timestep = int(model_num_timesteps(finished) or 0) if finished is not None else start_timestep
    timing_path = PPO_ROOT / "arm_timing.json"
    timing = json.loads(timing_path.read_text(encoding="utf-8")) if timing_path.is_file() else {
        "target_timesteps": TARGET_TIMESTEPS, "sessions": []}
    timing["sessions"].append({
        "started_at_utc": session_started_at.isoformat(),
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "elapsed_seconds": elapsed,
        "start_timestep": start_timestep,
        "end_timestep": end_timestep,
    })
    total_seconds = sum(s["elapsed_seconds"] for s in timing["sessions"])
    total_steps = max(end_timestep - timing["sessions"][0]["start_timestep"], 0)
    timing["total_training_seconds"] = total_seconds
    timing["end_timestep"] = end_timestep
    timing["steps_per_second"] = (total_steps / total_seconds) if total_seconds > 0 else None
    timing_path.write_text(json.dumps(timing, indent=2) + "\n", encoding="utf-8")
    print("arm_timing.json:", json.dumps({k: v for k, v in timing.items() if k != "sessions"}))


In [ ]:
from IPython.display import Markdown, display
artifact_paths = [PPO_ROOT / "run_config.json", PPO_ROOT / "arm_timing.json", PPO_ROOT / "evaluation_csv.csv", PPO_ROOT / "evaluation_scenarios.csv", PPO_ROOT / "holdout_selection.csv"]
for artifact in artifact_paths:
    if artifact.is_file():
        print(f"\n===== {artifact} =====")
        print(artifact.read_text(encoding="utf-8")[:20000])
    else:
        print(f"MISSING: {artifact}")
checkpoints = sorted((PPO_ROOT / "checkpoints").glob("*.sb3")) if (PPO_ROOT / "checkpoints").is_dir() else []
print("\ncheckpoints:", len(checkpoints), checkpoints[-1].name if checkpoints else "(none)")
print("Stage 2 resume command:", " ".join(ppo_command))
display(Markdown(f"**Experiment root:** `{EXPERIMENT_ROOT}`"))
